# Running on Modal

## What you'll learn

- Write a **tool op** — `ToolSpec` + `execute_command()` instead of `execute()`
- Deploy it once as a persistent Modal endpoint with `artisan modal deploy`
- Route steps to the endpoint by flipping `compute_provider="modal"`
- Understand the spawn/poll job model and per-artifact fan-out
- Authenticate clients with Modal proxy-auth tokens
- Debug endpoint execution

**Prerequisites:** [First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Compute Routing](01-compute-routing.ipynb).
**Estimated time:** 15 minutes
**GPU required:** No (the demo tool runs on CPU containers).


:::{note}
This tutorial requires a Modal account, Modal credentials, and a deployed
endpoint. Code cells are shown for reference and are not executed in the
docs build.
:::


In [1]:
from __future__ import annotations

from artisan.operations.examples import EchoTool
from artisan.orchestration import PipelineManager
from artisan.schemas.operation_config.compute import ComputeProvider, ModalComputeConfig
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [2]:
env = tutorial_setup("modal_execution", clean=True)
DELTA_ROOT = env.delta_root

## Tool operations

Modal compute runs **tool ops**: operations that wrap an external tool. A
tool op declares a `ToolSpec`, a `Params` model, and a `execute_command()` —
and **no** `execute_function()`; the framework provides one:

```python
class EchoTool(OperationDefinition):
    name = "echo_tool"
    tool = ToolSpec(executable="bash", interpreter=None)
    compute_provider = ComputeProvider(modal=ModalComputeConfig())

    class Params(BaseModel):
        model_config = {"extra": "forbid"}   # the endpoint's typed schema

        text: str = Field(default="hello from echo_tool", description="...")
        filename: str = Field(default="echo.txt", description="...")

    params: Params = Params()

    def execute_command(self, inputs: dict[str, Any]) -> list[str]:
        return [
            *self.tool.parts(),
            "-c",
            f'printf "%s\\n" "{self.params.text}" > "{self.params.filename}"',
        ]
```

`execute_command` is the single source of the command. Under
`compute_provider="local"` the framework runs it as a local subprocess;
under `"modal"` the deployed copy runs it in the tool's container. A tool
op's products are the files its command writes to the execute dir — memory
results and post-run glue belong in `postprocess()`.

Pure-Python operations (custom `execute_function()`) run on the local provider only.


## Deploy the endpoint (once per tool)

```bash
artisan modal deploy echo_tool
```

This deploys one persistent Modal app named `artisan-tool-echo_tool`: a
**worker** (the tool's image + hardware, one job per container) behind a
lightweight **HTTP endpoint** with a typed, tool-native API:

| Route       | Method | Purpose                                             |
| ----------- | ------ | --------------------------------------------------- |
| `/submit`   | POST   | `Params` JSON + input files (multipart) → `call_id` |
| `/result`   | GET    | Poll job status; returns the output manifest        |
| `/download` | GET    | Stream the output files as a tar                    |
| `/cancel`   | POST   | Terminate the running container                     |
| `/docs`     | GET    | Swagger UI                                          |

Deploy reads the op's **class-level** config: image, volumes, secrets, and
scaling from `ModalComputeConfig`; gpu/cpu/memory/timeout from
`ComputeResources`. Redeploy after changing the op's code, image, or
hardware.

The same endpoint serves Artisan pipelines and non-Artisan callers (curl,
other repositories) alike.


## Authentication

The endpoint requires Modal **proxy-auth tokens** (created in the Modal
dashboard under _Settings → Proxy Auth Tokens_). Clients send them as
`Modal-Key` / `Modal-Secret` headers.

The recommended setup is a `.env` file at the repo root — copy the
committed `.env.example` and fill in your token:

```bash
cp .env.example .env   # .env is gitignored
# then edit:
#   MODAL_PROXY_TOKEN_ID=wk-...
#   MODAL_PROXY_TOKEN_SECRET=ws-...
```

Artisan discovers the tokens from the process environment first, then the
nearest `.env` file walking up from the working directory — so Jupyter
kernels, cron jobs, and IDE test runners all work without shell-inherited
exports. Environment variables of the same names override the file (CI).

A different variable prefix can be configured per op via
`ModalComputeConfig.auth_secret`.


## Hardware lives on the deployment

Worker hardware is read from the op's class-level `ComputeResources` at
deploy time:

```python
class Op(OperationDefinition):
    ...
    compute_provider = ComputeProvider(modal=ModalComputeConfig(image=OP_IMAGE))
    compute_resources = ComputeResources(gpu="A100", memory_gb=32, timeout=7200)
```

Changing hardware means redeploying — a per-step `compute_resources`
override does **not** reconfigure an already-deployed endpoint.


In [3]:
config = ComputeProvider(active="modal", modal=ModalComputeConfig())

print(f"Active provider: {config.active}")
print(f"Available:       {config.available()}")
print(f"Worker image:    {config.modal.image}")
print(f"Poll interval:   {config.modal.poll_interval}s")

Active provider: modal
Available:       ['local', 'modal']
Worker image:    ghcr.io/dexterity-systems/artisan-worker:latest
Poll interval:   2.0s


## Running a step on the endpoint

`compute_provider="modal"` is the only change — the operation, params, and
output wiring stay identical:


In [4]:
pipeline = PipelineManager.create(
    name="modal_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_compute_provider="local",
)

# local first — same op, no Modal required
pipeline.run(
    operation=EchoTool,
    name="echo_local",
    params={"text": "ran locally", "filename": "local.txt"},
)

# then on the deployed endpoint — one argument changed
pipeline.run(
    operation=EchoTool,
    name="echo_modal",
    params={"text": "ran on Modal", "filename": "modal.txt"},
    compute_provider="modal",
)

summary = pipeline.finalize()
print(f"Pipeline complete: success={summary['overall_success']}")
inspect_pipeline(DELTA_ROOT)

19:51:29.489 | INFO    | artisan.orchestration.pipeline_manager - Pipeline 'modal_tutorial' initialized (run_id=modal_tutorial_20260611_025129_e3323bf1)

19:51:29.490 | INFO    | artisan.orchestration.pipeline_manager -   delta_root: /Users/ach/git/artisan/dev/docs/tutorials/07-compute-backends/runs/modal_execution/delta

19:51:29.491 | INFO    | artisan.orchestration.pipeline_manager -   staging_root: /Users/ach/git/artisan/dev/docs/tutorials/07-compute-backends/runs/modal_execution/staging

19:51:29.496 | INFO    | artisan.orchestration.pipeline_manager - Step 0 (echo_local) starting... [step_runner=local]

19:51:30.484 | INFO    | artisan.orchestration.engine.dispatch - Collected results from 1 futures

...

19:51:45.119 | INFO    | artisan.orchestration.pipeline_manager -   Step 0: echo_local       1.2s  [1/1]

19:51:45.119 | INFO    | artisan.orchestration.pipeline_manager -   Step 1: echo_modal       14.4s  [1/1]

19:51:45.120 | INFO    | artisan.orchestration.pipeline_manager -   Total: 15.6s

Pipeline complete: success=True


step,operation,status,produced,duration
i64,str,str,str,str
0,"""echo_local""","""ok""","""1 data""","""1.2s"""
1,"""echo_modal""","""ok""","""1 data""","""14.4s"""


## What happens during a modal step

For each artifact, the framework's `execute_function()`:

1. **Submits** the op's `Params` + input files to `/submit` (inline
   multipart; inputs that already live on object storage pass their
   `s3://` URI with zero re-upload)
2. **Polls** `/result` at `poll_interval` until the job leaves `pending`
3. **Downloads** the output tar into the artifact's `execute_dir` —
   recreating the exact layout of a local run (the tool log arrives
   separately and lands in the unit log, as it does locally)
4. Hands off to `postprocess`, lineage capture, and recording — the
   lifecycle is identical to local execution

A unit of N artifacts dispatches as N concurrent endpoint calls; the
worker runs one job per container (`max_inputs=1`), so Modal scales by
adding containers. Pipeline cancellation posts `/cancel`, which terminates
the running containers. The tool log arrives with the result (manifest
tail + tar), not streamed live.

**Transport limits:** inline inputs and the output tar are bounded at
100 MB per direction. Larger inputs should be `s3://` URIs; large static
data (model weights) belongs on Modal Volumes
(`ModalComputeConfig.volumes`), not in the request.


## Scale-out: watch containers count in parallel

`WaitTool` is a tool op built for exactly this: it counts up once per
second (`wait_tool [<container task id>] tick 3 / 10`), so you can _watch_ it run,
and it takes a `dataset` input role, so a step with eight artifacts fans
out as eight concurrent endpoint calls. Each container runs one job at a
time (`max_inputs=1`); Modal's autoscaler decides how many containers to
add as queue pressure grows — expect several for eight jobs, and set
`ModalComputeConfig.min_containers` to pre-warm a full-width pool.

Deploy it once:

```bash
artisan modal deploy wait_tool
```

Sequential, eight 10-second waits would take ~80 s. Fanned out, the step
finishes in a fraction of that (~35 s on a cold pool; less once warm).


In [5]:
from artisan.operations.examples import DataGenerator, WaitTool

scale = PipelineManager.create(
    name="modal_scale_out",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = scale.output

scale.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 8, "seed": 42},
)

# 8 artifacts -> 8 concurrent endpoint calls -> 8 containers
step = scale.run(
    operation=WaitTool,
    name="wait",
    inputs={"dataset": output("generate", "datasets")},
    params={"seconds": 10},
    compute_provider="modal",
)

scale.finalize()

19:51:57.872 | INFO    | artisan.orchestration.prefect_server - Prefect self-hosted: http://127.0.0.1:5202/api (source: env:PREFECT_API_URL)

19:51:57.873 | INFO    | artisan.orchestration.pipeline_manager - Pipeline 'modal_scale_out' initialized (run_id=modal_scale_out_20260611_025157_784b8773)

19:51:57.874 | INFO    | artisan.orchestration.pipeline_manager -   delta_root: /Users/ach/git/artisan/dev/docs/tutorials/07-compute-backends/runs/modal_execution/delta

19:51:57.875 | INFO    | artisan.orchestration.pipeline_manager -   staging_root: /Users/ach/git/artisan/dev/docs/tutorials/07-compute-backends/runs/modal_execution/staging

19:51:57.887 | INFO    | artisan.orchestration.pipeline_manager - Step 0 (generate) starting... [step_runner=local]

...

19:52:34.198 | INFO    | artisan.orchestration.pipeline_manager - Pipeline 'modal_scale_out' complete: 2 steps, all succeeded

19:52:34.199 | INFO    | artisan.orchestration.pipeline_manager -   Step 0: generate         1.1s  [1/1]

19:52:34.199 | INFO    | artisan.orchestration.pipeline_manager -   Step 1: wait             35.2s  [8/8]

19:52:34.199 | INFO    | artisan.orchestration.pipeline_manager -   Total: 36.3s

{'pipeline_name': 'modal_scale_out',
 'total_steps': 2,
 'steps': [{'step_number': 0,
   'name': 'generate',
   'success': True,
   'total': 1,
   'succeeded': 1,
   'failed': 0,
   'duration_seconds': 1.079299584031105},
  {'step_number': 1,
   'name': 'wait',
   'success': True,
   'total': 8,
   'succeeded': 8,
   'failed': 0,
   'duration_seconds': 35.21464483300224}],
 'overall_success': True}

### Watching it run

While the cell above executes (it takes ~20–60 s; the first run also pays
container cold starts):

- **Modal dashboard** — open the `artisan-tool-wait_tool` app: the
  container count climbs as the autoscaler fans out, and each container's log shows its ticks
  arriving one per second.
- **CLI** — in a terminal:

  ```bash
  modal app logs artisan-tool-wait_tool
  ```

  You'll see interleaved ticks from eight different container task ids
  counting up simultaneously — that interleaving _is_ the parallelism.

### Proof in the artifacts

Each container wrote its own task id into its marker file, so the
committed artifacts themselves record the fan-out — eight artifacts
across multiple distinct containers:


In [6]:
import polars as pl

markers = pl.read_delta(f"{env.delta_root}/artifacts/data")
rows = markers.filter(pl.col("content").bin.contains(b"seconds,host"))
hosts = set()
for content in rows["content"]:
    hosts.add(bytes(content).decode().splitlines()[1].split(",")[1])
print(f"distinct containers: {len(hosts)}")
print(sorted(hosts))

distinct containers: 4
['ta-01KTT9G20N3PZD2Z81P19PHK5C', 'ta-01KTT9G26DYRC8NTR9QDVBY6VR', 'ta-01KTT9G2PGF6R7SFSS125FA7ZC', 'ta-01KTT9G2YECRGFD2GPR87BZP53']


## Calling the endpoint without Artisan

The endpoint is a plain HTTP API — Artisan is one client among others:

```bash
URL=https://<workspace>--artisan-tool-echo-tool.modal.run

# submit
curl -X POST "$URL/submit" \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" \
  -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET" \
  -F 'params={"text": "hi from curl", "filename": "out.txt"}'
# → {"call_id": "fc-..."}

# poll
curl "$URL/result?call_id=fc-..." \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET"

# download outputs
curl -o outputs.tar "$URL/download?call_id=fc-..." \
  -H "Modal-Key: $MODAL_PROXY_TOKEN_ID" -H "Modal-Secret: $MODAL_PROXY_TOKEN_SECRET"
```


## Debugging

| Problem                            | Cause                                           | Fix                                                                             |
| ---------------------------------- | ----------------------------------------------- | ------------------------------------------------------------------------------- |
| `tool_endpoint_misconfigured`      | Op is not a tool op, or modal config missing    | Declare `ToolSpec` + `execute_command()`; configure `compute_provider.modal`      |
| "has no web URL — is it deployed?" | Endpoint not deployed                           | `artisan modal deploy <op>`                                                     |
| HTTP 407/401 at submit             | Missing or invalid proxy-auth tokens            | Set `MODAL_PROXY_TOKEN_ID` / `MODAL_PROXY_TOKEN_SECRET`                         |
| HTTP 422 at submit                 | Params don't match the op's schema              | The endpoint validates against `Params` (`extra="forbid"`)                      |
| `op_execute_failed`                | The tool exited non-zero                        | The error carries the stderr tail; full log in the parquet `tool_output` column |
| Result `expired`                   | Output polled more than 7 days after completion | Re-run the step                                                                 |

Develop locally first: run with `compute_provider="local"` until the
pipeline logic is correct, then flip to `"modal"`. Same op, same
`execute_command`, same outputs.


## Summary

| Concept                     | What it does                                                |
| --------------------------- | ----------------------------------------------------------- |
| Tool op                     | `ToolSpec` + `Params` + `execute_command()`; no `execute()`   |
| `artisan modal deploy <op>` | One-time deploy: persistent worker + HTTP endpoint per tool |
| `compute_provider="modal"`  | Route a step's per-artifact execute-phase calls to the endpoint |
| Spawn/poll                  | `/submit` → `call_id`; poll `/result`; `/download` outputs  |
| `ComputeResources`          | Worker hardware, read at deploy time                        |
| Proxy auth                  | `MODAL_PROXY_TOKEN_ID` / `MODAL_PROXY_TOKEN_SECRET` headers |
| Inline transport            | ≤100 MB per direction; `s3://` URIs bypass the bound        |

Operations, inputs, params, and output wiring are identical whether the
tool runs locally or on Modal. Results land in the same Delta Lake tables
regardless.


## Next steps

- [Compute Routing](01-compute-routing.ipynb) — Step runners vs compute providers
- [SLURM Execution](02-slurm-execution.ipynb) — Run operations on a SLURM cluster
- [Configure Execution](../../how-to-guides/configuring-execution.md) — Complete configuration reference
- [Execution Flow](../../concepts/execution-flow.md) — How the framework dispatches and tracks work
